# 27. The Integrated Berth & Crane Allocation Problem (BAP-QCAP)

## Tier 3: Particle Swarm Optimization (PSO) Metaheuristic

### Goal
Implement a Particle Swarm Optimization metaheuristic that explores the solution space using swarm intelligence to find high-quality berth and crane assignments, potentially surpassing simple heuristics while maintaining reasonable computational requirements.

### Key Assumptions
- PSO particles represent complete solutions (berth positions, start times, crane counts)
- Swarm intelligence guides search through particle communication and learning
- Continuous position values are decoded into feasible discrete assignments
- Constraint handling ensures spatial and temporal feasibility
- Population-based evolution balances exploration and exploitation

### Approach (Step-by-Step)
1. **Particle Representation**: Encode complete solutions as 3D vectors (position, time, cranes)
2. **Swarm Initialization**: Create random particle population with velocities
3. **Fitness Evaluation**: Decode particles and calculate objective values
4. **Constraint Handling**: Ensure feasibility through penalty functions
5. **PSO Evolution**: Update velocities and positions using cognitive/social components
6. **Convergence Tracking**: Monitor best solutions and swarm diversity

### What to Look for in the Results
- Improved solution quality over simple heuristics (typically 2-5% better)
- Convergence behavior showing swarm learning and optimization progress
- Balance between exploration (diverse solutions) and exploitation (refinement)
- Computational efficiency suitable for medium-sized instances

### Concrete Example (from the source)
PSO parameters: 50 particles, 200 iterations, w=0.7, c₁=c₂=1.5

Expected best solution:
- Vessel 1: Position 0m, Start 08:30, 3 cranes, Total time 20.1 hours
- Vessel 2: Position 300m, Start 10:00, 4 cranes, Total time 18.7 hours  
- Vessel 3: Position 600m, Start 12:15, 2 cranes, Total time 7.65 hours

PSO achieved 97.8% of optimal solution quality with convergence after 147 iterations.

In [1]:
from typing import Tuple, List, Dict, Set
from scipy import optimize
from itertools import product
from itertools import combinations

# Import required libraries for PSO implementation
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import Rectangle
import pandas as pd
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional
import seaborn as sns
from datetime import datetime, timedelta
import time
import random
from scipy.spatial.distance import euclidean

# Set random seeds for reproducibility
np.random.seed(42)
random.seed(42)

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully!")

Libraries imported successfully!


In [ ]:
# Define data structures for PSO algorithm
@dataclass
class Vessel:
    """Represents a vessel requiring berth and crane services"""
    id: int
    length: float  # meters
    workload: float  # TEU (twenty-foot equivalent units)
    arrival_time: float  # hours from start of day
    cost_per_hour: float = 1000.0  # cost per unit time
    max_cranes: int = 5  # maximum cranes that can serve this vessel
    
    def __post_init__(self):
        # Calculate priority based on workload and urgency
        time_window = max(1, 24 - self.arrival_time)
        self.priority = self.workload / (self.length * time_window)

@dataclass
class Particle:
    """Represents a particle in the PSO swarm"""
    position: np.ndarray  # continuous position vector
    velocity: np.ndarray  # velocity vector
    personal_best: np.ndarray  # best position found by this particle
    personal_best_fitness: float  # fitness of personal best
    current_fitness: float  # current fitness value
    
@dataclass
class PSOSolution:
    """Represents a decoded PSO solution"""
    vessel_id: int
    vessel: Vessel  # Reference to vessel object
    position: float  # berth position
    start_time: float  # service start time
    num_cranes: int  # number of cranes
    service_time: float  # service duration
    finish_time: float  # service finish time
    waiting_time: float  # waiting duration
    total_time: float  # total turnaround time
    cost: float  # total cost

@dataclass
class BAPQCAPInstance:
    """Problem instance containing all data for BAP-QCAP"""
    vessels: List[Vessel]
    quay_length: float  # total quay length in meters
    interference_factor: float  # productivity reduction per additional crane
    crane_productivity: float  # TEU per hour per crane
    planning_horizon: float  # hours

print("Data structures defined successfully!")

Data structures defined successfully!


In [ ]:
# Create the concrete example from the problem description
def create_concrete_example():
    """Create the example instance with 3 vessels from the problem statement"""
    
    # Define vessels with their characteristics
    vessels = [
        Vessel(id=1, length=300, workload=800, arrival_time=8.0),
        Vessel(id=2, length=300, workload=1200, arrival_time=10.0),
        Vessel(id=3, length=200, workload=400, arrival_time=12.0)
    ]
    
    # Create problem instance
    instance = BAPQCAPInstance(
        vessels=vessels,
        quay_length=1200,  # meters
        interference_factor=0.1,  # 10% productivity reduction per additional crane
        crane_productivity=30,  # 30 TEU/hour per crane
        planning_horizon=48  # hours
    )
    
    return instance

# Create and display the problem instance
instance = create_concrete_example()

print("=== Problem Instance Created ===")
print(f"Quay Length: {instance.quay_length}m")
print(f"Crane Productivity: {instance.crane_productivity} TEU/hour")
print(f"Interference Factor: {instance.interference_factor}")
print("\n=== Vessel Details ===")
for vessel in instance.vessels:
    print(f"Vessel {vessel.id}: {vessel.length}m, {vessel.workload} TEU, Arrival: {vessel.arrival_time}:00")
    print(f"  Priority: {vessel.priority:.2f}, Max Cranes: {vessel.max_cranes}")

=== Problem Instance Created ===
Quay Length: 1200m
Crane Productivity: 30 TEU/hour
Interference Factor: 0.1

=== Vessel Details ===
Vessel 1: 300m, 800 TEU, Arrival: 8.0:00
  Priority: 0.17, Max Cranes: 5
Vessel 2: 300m, 1200 TEU, Arrival: 10.0:00
  Priority: 0.29, Max Cranes: 5
Vessel 3: 200m, 400 TEU, Arrival: 12.0:00
  Priority: 0.17, Max Cranes: 5


In [ ]:
class PSOBerthingOptimizer:
    """Particle Swarm Optimization for BAP-QCAP"""
    
    def __init__(self, instance: BAPQCAPInstance, swarm_size: int = 50):
        self.instance = instance
        self.swarm_size = swarm_size
        self.num_vessels = len(instance.vessels)
        
        # Each vessel needs 3 dimensions: position, start_time, crane_count
        self.dimension = self.num_vessels * 3
        
        # PSO parameters (from the problem description)
        self.w = 0.7  # inertia weight
        self.c1 = 1.5  # cognitive parameter
        self.c2 = 1.5  # social parameter
        
        # Initialize swarm
        self.particles = []
        self.global_best_position = None
        self.global_best_fitness = float('inf')
        
        # Convergence tracking
        self.convergence_history = []
        self.diversity_history = []
        
        self.initialize_swarm()
    
    def initialize_swarm(self):
        """Initialize the PSO swarm with random particles"""
        print(f"Initializing PSO swarm with {self.swarm_size} particles...")
        
        for i in range(self.swarm_size):
            # Create random particle
            position = self.create_random_position()
            velocity = np.random.uniform(-1, 1, self.dimension)
            
            # Evaluate fitness
            fitness = self.evaluate_fitness(position)
            
            # Create particle
            particle = Particle(
                position=position,
                velocity=velocity,
                personal_best=position.copy(),
                personal_best_fitness=fitness,
                current_fitness=fitness
            )
            
            self.particles.append(particle)
            
            # Update global best
            if fitness < self.global_best_fitness:
                self.global_best_fitness = fitness
                self.global_best_position = position.copy()
        
        print(f"Initial global best fitness: {self.global_best_fitness:.2f}")
    
    def create_random_position(self) -> np.ndarray:
        """Create a random feasible position vector"""
        position = []
        
        for vessel in self.instance.vessels:
            # Random berth position (0 to quay_length - vessel_length)
            berth_pos = np.random.uniform(0, self.instance.quay_length - vessel.length)
            
            # Random start time (arrival_time to arrival_time + 24h)
            start_time = np.random.uniform(vessel.arrival_time, vessel.arrival_time + 24)
            
            # Random crane count (1 to max_cranes)
            crane_count = np.random.uniform(1, vessel.max_cranes)
            
            position.extend([berth_pos, start_time, crane_count])
        
        return np.array(position)
    
    def decode_particle(self, position: np.ndarray) -> List[PSOSolution]:
        """Decode particle position into feasible BAP-QCAP solution"""
        solutions = []
        
        for i, vessel in enumerate(self.instance.vessels):
            idx = i * 3
            
            # Extract and clamp values to feasible ranges
            berth_pos = max(0, min(position[idx], self.instance.quay_length - vessel.length))
            start_time = max(vessel.arrival_time, position[idx + 1])
            crane_count = max(1, min(int(position[idx + 2]), vessel.max_cranes))
            
            # Calculate service time
            service_time = self.calculate_service_time(vessel, crane_count)
            finish_time = start_time + service_time
            waiting_time = start_time - vessel.arrival_time
            total_time = waiting_time + service_time
            cost = total_time * vessel.cost_per_hour
            
            solution = PSOSolution(
                vessel_id=vessel.id,
                vessel=vessel,  # Add vessel reference
                position=berth_pos,
                start_time=start_time,
                num_cranes=crane_count,
                service_time=service_time,
                finish_time=finish_time,
                waiting_time=waiting_time,
                total_time=total_time,
                cost=cost
            )
            
            solutions.append(solution)
        
        return solutions
    
    def calculate_service_time(self, vessel: Vessel, num_cranes: int) -> float:
        """Calculate service time considering crane interference"""
        if num_cranes == 0:
            return float('inf')
            
        # Total productivity with interference
        total_productivity = 0
        for i in range(num_cranes):
            productivity_factor = 1 - self.instance.interference_factor * i
            total_productivity += self.instance.crane_productivity * productivity_factor
        
        # Service time = workload / total productivity
        service_time = vessel.workload / total_productivity
        return service_time
    
    def is_feasible(self, solutions: List[PSOSolution]) -> bool:
        """Check if the decoded solution is feasible"""
        # Check for spatial conflicts
        for i in range(len(solutions)):
            for j in range(i + 1, len(solutions)):
                sol_i = solutions[i]
                sol_j = solutions[j]
                
                # Check spatial overlap
                spatial_overlap = (sol_i.position < sol_j.position + sol_j.vessel.length and 
                                 sol_j.position < sol_i.position + sol_i.vessel.length)
                
                if spatial_overlap:
                    # Check temporal overlap
                    temporal_overlap = (sol_i.start_time < sol_j.finish_time and 
                                     sol_j.start_time < sol_i.finish_time)
                    
                    if temporal_overlap:
                        return False  # Conflict found
        
        return True  # No conflicts
    
    def evaluate_fitness(self, position: np.ndarray) -> float:
        """Evaluate fitness of a particle position"""
        solutions = self.decode_particle(position)
        
        # Check feasibility
        if not self.is_feasible(solutions):
            # Apply penalty for infeasibility
            penalty = 10000
            return sum(sol.cost for sol in solutions) + penalty
        
        # Calculate total cost
        total_cost = sum(sol.cost for sol in solutions)
        return total_cost
    
    def calculate_swarm_diversity(self) -> float:
        """Calculate diversity of the swarm"""
        if len(self.particles) <= 1:
            return 0.0
        
        positions = np.array([p.position for p in self.particles])
        centroid = np.mean(positions, axis=0)
        
        # Calculate average distance from centroid
        distances = [euclidean(p.position, centroid) for p in self.particles]
        diversity = np.mean(distances)
        
        return diversity
    
    def update_particles(self):
        """Update particle velocities and positions"""
        for particle in self.particles:
            # Generate random coefficients
            r1 = np.random.uniform(0, 1, self.dimension)
            r2 = np.random.uniform(0, 1, self.dimension)
            
            # Update velocity
            cognitive_component = self.c1 * r1 * (particle.personal_best - particle.position)
            social_component = self.c2 * r2 * (self.global_best_position - particle.position)
            
            particle.velocity = (self.w * particle.velocity + 
                                cognitive_component + 
                                social_component)
            
            # Update position
            particle.position += particle.velocity
            
            # Evaluate new fitness
            particle.current_fitness = self.evaluate_fitness(particle.position)
            
            # Update personal best
            if particle.current_fitness < particle.personal_best_fitness:
                particle.personal_best = particle.position.copy()
                particle.personal_best_fitness = particle.current_fitness
                
                # Update global best
                if particle.current_fitness < self.global_best_fitness:
                    self.global_best_fitness = particle.current_fitness
                    self.global_best_position = particle.position.copy()
    
    def optimize(self, max_iterations: int = 200) -> Tuple[List[PSOSolution], Dict]:
        """Run PSO optimization"""
        print(f"\n=== Starting PSO Optimization ===")
        print(f"Swarm size: {self.swarm_size}")
        print(f"Max iterations: {max_iterations}")
        print(f"Problem dimension: {self.dimension}")
        
        start_time = time.time()
        
        for iteration in range(max_iterations):
            # Update particles
            self.update_particles()
            
            # Record convergence
            self.convergence_history.append(self.global_best_fitness)
            self.diversity_history.append(self.calculate_swarm_diversity())
            
            # Progress reporting
            if iteration % 50 == 0 or iteration == max_iterations - 1:
                print(f"Iteration {iteration:3d}: Best Fitness = {self.global_best_fitness:.2f}, "
                      f"Diversity = {self.diversity_history[-1]:.2f}")
        
        end_time = time.time()
        computation_time = end_time - start_time
        
        # Decode best solution
        best_solution = self.decode_particle(self.global_best_position)
        
        # Prepare results
        results = {
            'best_fitness': self.global_best_fitness,
            'computation_time': computation_time,
            'iterations': max_iterations,
            'convergence_history': self.convergence_history,
            'diversity_history': self.diversity_history,
            'final_diversity': self.diversity_history[-1],
            'convergence_iteration': self.find_convergence_iteration()
        }
        
        print(f"\n=== PSO Optimization Complete ===")
        print(f"Final best fitness: {self.global_best_fitness:.2f}")
        print(f"Computation time: {computation_time:.2f} seconds")
        print(f"Convergence at iteration: {results['convergence_iteration']}")
        
        return best_solution, results
    
    def find_convergence_iteration(self) -> int:
        """Find the iteration where convergence occurred"""
        if len(self.convergence_history) < 10:
            return len(self.convergence_history) - 1
        
        # Look for the point where improvement becomes minimal
        for i in range(10, len(self.convergence_history)):
            recent_improvement = (self.convergence_history[i-10] - self.convergence_history[i]) / self.convergence_history[i-10]
            if recent_improvement < 0.01:  # Less than 1% improvement in 10 iterations
                return i
        
        return len(self.convergence_history) - 1

print("PSO optimizer class defined successfully!")

PSO optimizer class defined successfully!


In [ ]:
# Run PSO optimization on the BAP-QCAP instance
def run_pso_optimization(instance):
    """Run PSO optimization and return results"""
    
    # Create PSO optimizer
    pso = PSOBerthingOptimizer(instance, swarm_size=50)
    
    # Run optimization
    best_solution, results = pso.optimize(max_iterations=200)
    
    return best_solution, results, pso

# Execute PSO optimization
best_solution, pso_results, pso_optimizer = run_pso_optimization(instance)

print("\n=== Best PSO Solution ===")
for solution in best_solution:
    vessel = solution.vessel  # Use the vessel reference
    print(f"Vessel {solution.vessel_id}:")
    print(f"  Position: {solution.position:.1f}m - {solution.position + vessel.length:.1f}m")
    print(f"  Start: {solution.start_time:.1f}:00, Finish: {solution.finish_time:.1f}:00")
    print(f"  Cranes: {solution.num_cranes}, Service: {solution.service_time:.1f}h")
    print(f"  Waiting: {solution.waiting_time:.1f}h, Total: {solution.total_time:.1f}h")
    print(f"  Cost: ${solution.cost:.0f}")
    print()

Initializing PSO swarm with 50 particles...
Initial global best fitness: 57791.41

=== Starting PSO Optimization ===
Swarm size: 50
Max iterations: 200
Problem dimension: 9
Iteration   0: Best Fitness = 35877.16, Diversity = 326.73
Iteration  50: Best Fitness = 20000.00, Diversity = 432.47


In [ ]:
# Create comprehensive PSO visualizations
def visualize_pso_results(instance, best_solution, pso_results, pso_optimizer):
    """Create detailed visualizations of PSO optimization results"""
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Particle Swarm Optimization Results', fontsize=16, fontweight='bold')
    
    # 1. Berth Layout Visualization
    ax1 = axes[0, 0]
    ax1.set_title('PSO Optimal Berth Allocation', fontweight='bold')
    ax1.set_xlabel('Berth Position (meters)')
    ax1.set_ylabel('Vessel ID')
    
    colors = plt.cm.Set3(np.linspace(0, 1, len(best_solution)))
    
    for i, solution in enumerate(best_solution):
        vessel = solution.vessel
        
        # Draw vessel rectangle
        rect = Rectangle((solution.position, solution.vessel_id - 0.4), 
                        vessel.length, 0.8, 
                        facecolor=colors[i], edgecolor='black', linewidth=2)
        ax1.add_patch(rect)
        
        # Add vessel label
        ax1.text(solution.position + vessel.length/2, solution.vessel_id, 
                f'V{solution.vessel_id}\n{solution.num_cranes} cranes\nPSO',
                ha='center', va='center', fontweight='bold', fontsize=9)
    
    ax1.set_xlim(-50, instance.quay_length + 50)
    ax1.set_ylim(0, len(best_solution) + 1)
    ax1.grid(True, alpha=0.3)
    
    # 2. Convergence History
    ax2 = axes[0, 1]
    ax2.set_title('PSO Convergence History', fontweight='bold')
    ax2.set_xlabel('Iteration')
    ax2.set_ylabel('Best Fitness (Cost)')
    
    iterations = range(len(pso_results['convergence_history']))
    ax2.plot(iterations, pso_results['convergence_history'], 'b-', linewidth=2, label='Best Fitness')
    
    # Mark convergence point
    conv_iter = pso_results['convergence_iteration']
    conv_fitness = pso_results['convergence_history'][conv_iter]
    ax2.axvline(x=conv_iter, color='red', linestyle='--', alpha=0.7, label=f'Convergence ({conv_iter})')
    ax2.scatter([conv_iter], [conv_fitness], color='red', s=100, zorder=5)
    
    ax2.grid(True, alpha=0.3)
    ax2.legend()
    
    # 3. Swarm Diversity
    ax3 = axes[1, 0]
    ax3.set_title('Swarm Diversity Over Time', fontweight='bold')
    ax3.set_xlabel('Iteration')
    ax3.set_ylabel('Swarm Diversity')
    
    iterations = range(len(pso_results['diversity_history']))
    ax3.plot(iterations, pso_results['diversity_history'], 'g-', linewidth=2)
    ax3.fill_between(iterations, pso_results['diversity_history'], alpha=0.3, color='green')
    
    # Mark convergence point
    ax3.axvline(x=conv_iter, color='red', linestyle='--', alpha=0.7)
    ax3.text(conv_iter + 5, max(pso_results['diversity_history']) * 0.8, 
            f'Convergence\n({conv_iter} iter)', fontsize=9, color='red')
    
    ax3.grid(True, alpha=0.3)
    
    # 4. Performance Metrics Dashboard
    ax4 = axes[1, 1]
    ax4.set_title('PSO Performance Metrics', fontweight='bold')
    ax4.axis('off')
    
    # Calculate metrics
    total_cost = sum(sol.cost for sol in best_solution)
    total_time = sum(sol.total_time for sol in best_solution)
    avg_time = total_time / len(best_solution)
    total_cranes = sum(sol.num_cranes for sol in best_solution)
    
    metrics_text = f"""
🎯 OPTIMIZATION RESULTS
─────────────────────
Total Cost: ${total_cost:,.0f}
Avg Turnaround: {avg_time:.1f}h
Total Cranes: {total_cranes}
Best Fitness: {pso_results['best_fitness']:.0f}

⚡ ALGORITHM PERFORMANCE
─────────────────────
Swarm Size: {pso_optimizer.swarm_size}
Iterations: {pso_results['iterations']}
Convergence: {pso_results['convergence_iteration']}
Comp Time: {pso_results['computation_time']:.2f}s
Final Diversity: {pso_results['final_diversity']:.2f}

🔧 PSO PARAMETERS
─────────────────────
Inertia (w): {pso_optimizer.w}
Cognitive (c₁): {pso_optimizer.c1}
Social (c₂): {pso_optimizer.c2}
Dimension: {pso_optimizer.dimension}
Problem: BAP-QCAP
"""
    
    ax4.text(0.1, 0.9, metrics_text, transform=ax4.transAxes, fontsize=10,
             verticalalignment='top', fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))
    
    plt.tight_layout()
    plt.show()
    
    # Print detailed analysis
    print("\n=== PSO Performance Analysis ===")
    print(f"Solution Quality: ${total_cost:,.0f} total cost")
    print(f"Average Turnaround: {avg_time:.1f} hours per vessel")
    print(f"Crane Utilization: {total_cranes} cranes assigned")
    print(f"Convergence Speed: {pso_results['convergence_iteration']} iterations")
    print(f"Computational Efficiency: {pso_results['computation_time']:.2f} seconds")
    print(f"Swarm Diversity: {pso_results['final_diversity']:.2f} (exploration capability)")

# Visualize PSO results
visualize_pso_results(instance, best_solution, pso_results, pso_optimizer)

MINIMUM SPANNING TREE PROBLEM - MATHEMATICAL FORMULATION

Problem: 8-city distribution network
Cities: A, B, C, D, E, F, G, H
Total possible connections: 11
Solution method: MILP
Optimal: True

OPTIMAL SOLUTION

Total Minimum Cost: $36 million
Number of edges selected: 7

Selected edges (in order of cost):
  1. E -- G: $3 million
  2. D -- E: $4 million
  3. B -- C: $5 million
  4. F -- G: $5 million
  5. A -- D: $6 million
  6. F -- H: $6 million
  7. B -- H: $7 million

COST ANALYSIS
Total cost of all connections: $73 million
MST cost: $36 million
Cost savings: $37 million
Network efficiency: 49.3%

Edge cost distribution:
  Low (0-4): 1 edges
  Medium (4-7): 5 edges
  High (7+): 1 edges

CONNECTIVITY ANALYSIS

Node degrees (number of connections):
  A: 1 connections
  B: 2 connections
  C: 1 connections
  D: 2 connections
  E: 2 connections
  F: 2 connections
  G: 2 connections
  H: 2 connections

Degree range: 1 to 2

SOLUTION VERIFICATION
Expected total cost: $36 million
Actual to

In [ ]:
# Compare PSO with heuristic baseline
def compare_pso_with_heuristic(instance, best_solution, pso_results):
    """Compare PSO results with priority-based heuristic"""
    
    print("=== PSO vs Heuristic Comparison ===")
    
    # Implement simple heuristic for comparison
    def simple_heuristic_solution(instance):
        """Generate a simple FCFS heuristic solution"""
        solutions = []
        current_time = 0
        
        # Sort vessels by arrival time
        sorted_vessels = sorted(instance.vessels, key=lambda v: v.arrival_time)
        
        for vessel in sorted_vessels:
            # Simple assignment
            position = current_time * 100  # Simple spacing
            position = min(position, instance.quay_length - vessel.length)
            
            start_time = max(vessel.arrival_time, current_time)
            num_cranes = 3  # Fixed crane assignment
            
            # Calculate service time
            total_productivity = 0
            for i in range(num_cranes):
                productivity_factor = 1 - instance.interference_factor * i
                total_productivity += instance.crane_productivity * productivity_factor
            
            service_time = vessel.workload / total_productivity
            finish_time = start_time + service_time
            waiting_time = start_time - vessel.arrival_time
            total_time = waiting_time + service_time
            cost = total_time * vessel.cost_per_hour
            
            solution = PSOSolution(
                vessel_id=vessel.id,
                vessel=vessel,
                position=position,
                start_time=start_time,
                num_cranes=num_cranes,
                service_time=service_time,
                finish_time=finish_time,
                waiting_time=waiting_time,
                total_time=total_time,
                cost=cost
            )
            
            solutions.append(solution)
            current_time = finish_time
        
        return solutions
    
    # Generate heuristic solution
    heuristic_solutions = simple_heuristic_solution(instance)
    
    # Calculate comparison metrics
    pso_cost = sum(sol.cost for sol in best_solution)
    pso_time = sum(sol.total_time for sol in best_solution)
    pso_cranes = sum(sol.num_cranes for sol in best_solution)
    
    heuristic_cost = sum(sol.cost for sol in heuristic_solutions)
    heuristic_time = sum(sol.total_time for sol in heuristic_solutions)
    heuristic_cranes = sum(sol.num_cranes for sol in heuristic_solutions)
    
    # Calculate improvement percentages
    cost_improvement = (heuristic_cost - pso_cost) / heuristic_cost * 100
    time_improvement = (heuristic_time - pso_time) / heuristic_time * 100
    
    print(f"\nComparison Results:")
    print(f"{'Metric':<20} {'PSO':<15} {'Heuristic':<15} {'Improvement':<15}")
    print("-" * 65)
    print(f"{'Total Cost ($)':<20} {pso_cost:>13,.0f} {heuristic_cost:>13,.0f} {cost_improvement:>13.1f}%")
    print(f"{'Total Time (h)':<20} {pso_time:>13.1f} {heuristic_time:>13.1f} {time_improvement:>13.1f}%")
    print(f"{'Total Cranes':<20} {pso_cranes:>13} {heuristic_cranes:>13} {'N/A':>13}")
    
    # Create comparison visualization
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle('PSO vs Heuristic Performance Comparison', fontsize=16, fontweight='bold')
    
    # Cost comparison
    methods = ['PSO', 'Heuristic']
    costs = [pso_cost, heuristic_cost]
    colors = ['blue', 'orange']
    
    bars1 = ax1.bar(methods, costs, color=colors, alpha=0.7)
    ax1.set_ylabel('Total Cost ($)')
    ax1.set_title('Cost Comparison')
    ax1.grid(True, alpha=0.3)
    
    for bar, cost in zip(bars1, costs):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1000,
                f'${cost:,.0f}', ha='center', va='bottom', fontweight='bold')
    
    # Time comparison
    times = [pso_time, heuristic_time]
    
    bars2 = ax2.bar(methods, times, color=colors, alpha=0.7)
    ax2.set_ylabel('Total Turnaround Time (hours)')
    ax2.set_title('Time Comparison')
    ax2.grid(True, alpha=0.3)
    
    for bar, time_val in zip(bars2, times):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{time_val:.1f}h', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    return {
        'pso_cost': pso_cost,
        'heuristic_cost': heuristic_cost,
        'cost_improvement': cost_improvement,
        'pso_time': pso_time,
        'heuristic_time': heuristic_time,
        'time_improvement': time_improvement
    }

# Run comparison
comparison_results = compare_pso_with_heuristic(instance, best_solution, pso_results)

=== PSO vs Heuristic Comparison ===

Comparison Results:
Metric               PSO             Heuristic       Improvement    
-----------------------------------------------------------------
Total Cost ($)              20,000        58,198          65.6%
Total Time (h)                20.0          58.2          65.6%
Total Cranes                    15             9           N/A


In [ ]:
try:
    # Parameter sensitivity analysis
    def parameter_sensitivity_analysis(instance):
        """Analyze PSO performance with different parameter settings"""
        
        print("=== PSO Parameter Sensitivity Analysis ===")
        
        # Test different parameter combinations
        parameter_sets = [
            {'w': 0.7, 'c1': 1.5, 'c2': 1.5, 'name': 'Balanced'},
            {'w': 0.9, 'c1': 2.0, 'c2': 1.0, 'name': 'Exploration'},
            {'w': 0.5, 'c1': 1.0, 'c2': 2.0, 'name': 'Exploitation'},
            {'w': 0.8, 'c1': 1.8, 'c2': 1.8, 'name': 'Aggressive'}
        ]
        
        results = []
        
        for params in parameter_sets:
            print(f"\nTesting {params['name']} parameters: w={params['w']}, c1={params['c1']}, c2={params['c2']}")
            
            # Create PSO with custom parameters
            pso = PSOBerthingOptimizer(instance, swarm_size=30)  # Smaller swarm for faster testing
            pso.w = params['w']
            pso.c1 = params['c1']
            pso.c2 = params['c2']
            
            # Run optimization
            best_solution, pso_results, _ = run_pso_optimization(instance)
            
            # Record results
            total_cost = sum(sol.cost for sol in best_solution)
            
            results.append({
                'name': params['name'],
                'w': params['w'],
                'c1': params['c1'],
                'c2': params['c2'],
                'best_cost': total_cost,
                'convergence_iter': pso_results['convergence_iteration'],
                'final_diversity': pso_results['final_diversity'],
                'comp_time': pso_results['computation_time']
            })
            
            print(f"  Best cost: ${total_cost:,.0f}")
            print(f"  Convergence: {pso_results['convergence_iteration']} iterations")
        
        # Create sensitivity visualization
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        fig.suptitle('PSO Parameter Sensitivity Analysis', fontsize=16, fontweight='bold')
        
        names = [r['name'] for r in results]
        
        # Plot 1: Solution Quality
        ax1 = axes[0, 0]
        costs = [r['best_cost'] for r in results]
        bars1 = ax1.bar(names, costs, color=['blue', 'green', 'orange', 'red'], alpha=0.7)
        ax1.set_ylabel('Best Cost ($)')
        ax1.set_title('Solution Quality by Parameter Set')
        ax1.grid(True, alpha=0.3)
        
        for bar, cost in zip(bars1, costs):
            ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
                    f'${cost:,.0f}', ha='center', va='bottom', fontweight='bold')
        
        # Plot 2: Convergence Speed
        ax2 = axes[0, 1]
        conv_iters = [r['convergence_iter'] for r in results]
        bars2 = ax2.bar(names, conv_iters, color=['blue', 'green', 'orange', 'red'], alpha=0.7)
        ax2.set_ylabel('Convergence Iteration')
        ax2.set_title('Convergence Speed')
        ax2.grid(True, alpha=0.3)
        
        for bar, conv_iter in zip(bars2, conv_iters):
            ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                    f'{conv_iter}', ha='center', va='bottom', fontweight='bold')
        
        # Plot 3: Final Diversity
        ax3 = axes[1, 0]
        diversities = [r['final_diversity'] for r in results]
        bars3 = ax3.bar(names, diversities, color=['blue', 'green', 'orange', 'red'], alpha=0.7)
        ax3.set_ylabel('Final Swarm Diversity')
        ax3.set_title('Exploration Capability')
        ax3.grid(True, alpha=0.3)
        
        # Plot 4: Parameter Space Visualization
        ax4 = axes[1, 1]
        ax4.set_title('Parameter Space Comparison')
        ax4.set_xlabel('Cognitive Parameter (c₁)')
        ax4.set_ylabel('Social Parameter (c₂)')
        
        for i, result in enumerate(results):
            ax4.scatter(result['c1'], result['c2'], s=200, c=['blue', 'green', 'orange', 'red'][i], 
                       edgecolors='black', linewidth=2, label=result['name'])
            ax4.annotate(result['name'], (result['c1'], result['c2']), 
                        xytext=(5, 5), textcoords='offset points', fontweight='bold')
        
        ax4.set_xlim(0, 2.5)
        ax4.set_ylim(0, 2.5)
        ax4.grid(True, alpha=0.3)
        ax4.legend()
        
        plt.tight_layout()
        plt.show()
        
        # Print summary table
        print("\nParameter Sensitivity Summary:")
        print("Parameter Set | Cost ($) | Conv Iter | Diversity | Time (s)")
        print("-" * 65)
        for result in results:
            print(f"{result['name']:<12} | {result['best_cost']:>8.0f} | {result['convergence_iter']:>9} | {result['final_diversity']:>8.2f} | {result['comp_time']:>7.2f}")
        
        return results
    
    # Run parameter sensitivity analysis
    sensitivity_results = parameter_sensitivity_analysis(instance)
except Exception as e:
    print(f'Error in cell: {e}')

[Timeout - cell wrapped]

### Why This Tier Exists vs Previous Tiers

The Particle Swarm Optimization tier addresses key limitations of both mathematical formulation and simple heuristics:

- **Solution Quality**: Typically achieves 2-5% better solutions than simple heuristics by exploring multiple solution regions
- **Global Search**: Swarm intelligence avoids local optima that trap greedy heuristics
- **Balanced Exploration**: PSO maintains diversity while converging toward high-quality solutions
- **Adaptive Learning**: Particles learn from both personal experience and swarm knowledge
- **Computational Efficiency**: Much faster than exact methods while maintaining solution quality

### Pros vs Cons

**Advantages:**
- ✅ **High Solution Quality**: Often within 2-5% of optimal, better than simple heuristics
- ✅ **Global Optimization**: Swarm intelligence explores multiple solution regions
- ✅ **Fast Convergence**: Typically converges in 100-200 iterations
- ✅ **Few Parameters**: Only inertia, cognitive, and social parameters to tune
- ✅ **Scalable**: Handles medium to large instances efficiently

**Disadvantages:**
- ❌ **No Optimality Guarantee**: Cannot guarantee global optimality
- ❌ **Parameter Sensitivity**: Performance depends on parameter tuning
- ❌ **Stochastic Nature**: Different runs may produce different results
- ❌ **Constraint Handling**: Requires penalty functions for complex constraints
- ❌ **Premature Convergence**: Risk of converging to local optima

### When to Use This Tier

Use PSO when:
- **Medium-Sized Instances**: 10-50 vessels where MIP is too slow but heuristics may be suboptimal
- **Solution Quality Critical**: When 2-5% improvement over heuristics is valuable
- **Multiple Runs Acceptable**: Can run multiple times to find best solution
- **Parameter Tuning Possible**: Have time to calibrate PSO parameters
- **Complex Solution Space**: Problem has many local optima where heuristics get trapped

For guaranteed optimality, use Tier 1. For real-time decisions, use Tier 2.

In [ ]:
# Final summary and validation
def final_pso_summary(instance, best_solution, pso_results, comparison_results):
    """Provide comprehensive summary of PSO optimization"""
    
    print("=== PARTICLE SWARM OPTIMIZATION SUMMARY ===")
    print("\n🔬 ALGORITHM CHARACTERISTICS:")
    print(f"  • Method: Particle Swarm Optimization (PSO)")
    print(f"  • Swarm Size: {pso_optimizer.swarm_size} particles")
    print(f"  • Problem Dimension: {pso_optimizer.dimension} (3 variables per vessel)")
    print(f"  • Inertia Weight (w): {pso_optimizer.w}")
    print(f"  • Cognitive Parameter (c₁): {pso_optimizer.c1}")
    print(f"  • Social Parameter (c₂): {pso_optimizer.c2}")
    
    print("\n📊 OPTIMIZATION RESULTS:")
    total_cost = sum(sol.cost for sol in best_solution)
    total_time = sum(sol.total_time for sol in best_solution)
    avg_time = total_time / len(best_solution)
    total_cranes = sum(sol.num_cranes for sol in best_solution)
    
    print(f"  • Total Cost: ${total_cost:,.0f}")
    print(f"  • Average Turnaround: {avg_time:.1f} hours")
    print(f"  • Total Cranes Assigned: {total_cranes}")
    print(f"  • Convergence Iteration: {pso_results['convergence_iteration']}")
    print(f"  • Final Swarm Diversity: {pso_results['final_diversity']:.2f}")
    
    print("\n⚡ PERFORMANCE METRICS:")
    print(f"  • Computation Time: {pso_results['computation_time']:.2f} seconds")
    print(f"  • Convergence Speed: {pso_results['convergence_iteration']} iterations")
    print(f"  • Solution Quality: {comparison_results['cost_improvement']:.1f}% better than heuristic")
    print(f"  • Time Improvement: {comparison_results['time_improvement']:.1f}% faster than heuristic")
    print(f"  • Algorithm Efficiency: {pso_results['iterations']/pso_results['computation_time']:.0f} iter/second")
    
    print("\n🧠 SWARM INTELLIGENCE INSIGHTS:")
    print("  • Particles learn from both personal and social experience")
    print("  • Balance between exploration (diversity) and exploitation (convergence)")
    print("  • Continuous position encoding enables smooth solution space navigation")
    print("  • Constraint handling through penalty functions maintains feasibility")
    
    print("\n⚠️ ALGORITHM LIMITATIONS:")
    print("  • No guarantee of global optimality")
    print("  • Performance sensitive to parameter tuning")
    print("  • Stochastic nature produces variable results")
    print("  • May converge prematurely to local optima")
    
    print("\n📋 DETAILED SOLUTION ASSIGNMENTS:")
    for solution in best_solution:
        vessel = solution.vessel
        print(f"  Vessel {solution.vessel_id}: Pos {solution.position:.0f}m, ")
        print(f"    Start {solution.start_time:.1f}:00, {solution.num_cranes} cranes")
        print(f"    Service {solution.service_time:.1f}h, Total {solution.total_time:.1f}h")
    
    print("\n🎯 OPERATIONAL RECOMMENDATIONS:")
    print("  • Use for medium-sized instances (10-50 vessels)")
    print("  • Run multiple times and keep best solution")
    print("  • Calibrate parameters based on problem characteristics")
    print("  • Monitor convergence to avoid premature stagnation")
    print("  • Consider hybrid approaches with local refinement")
    
    print("\n🔬 REPLICATION GUIDELINES:")
    print("  • Set random seeds for reproducible results")
    print("  • Use 30-50 particles for good balance of speed/quality")
    print("  • Standard parameters: w=0.7, c₁=1.5, c₂=1.5 work well for BAP-QCAP")
    print("  • 200 iterations typically sufficient for convergence")
    print("  • Monitor swarm diversity to ensure adequate exploration")

# Generate final summary
final_pso_summary(instance, best_solution, pso_results, comparison_results)

=== PARTICLE SWARM OPTIMIZATION SUMMARY ===

🔬 ALGORITHM CHARACTERISTICS:
  • Method: Particle Swarm Optimization (PSO)
  • Swarm Size: 50 particles
  • Problem Dimension: 9 (3 variables per vessel)
  • Inertia Weight (w): 0.7
  • Cognitive Parameter (c₁): 1.5
  • Social Parameter (c₂): 1.5

📊 OPTIMIZATION RESULTS:
  • Total Cost: $20,000
  • Average Turnaround: 6.7 hours
  • Total Cranes Assigned: 15
  • Convergence Iteration: 11
  • Final Swarm Diversity: 490.53

⚡ PERFORMANCE METRICS:
  • Computation Time: 2.14 seconds
  • Convergence Speed: 11 iterations
  • Solution Quality: 65.6% better than heuristic
  • Time Improvement: 65.6% faster than heuristic
  • Algorithm Efficiency: 93 iter/second

🧠 SWARM INTELLIGENCE INSIGHTS:
  • Particles learn from both personal and social experience
  • Balance between exploration (diversity) and exploitation (convergence)
  • Continuous position encoding enables smooth solution space navigation
  • Constraint handling through penalty functions ma